# Wan2.1-T2V-1.3B: Baseline Inference

This notebook establishes a reproducible baseline for
text-to-video generation using Wan2.1-T2V-1.3B.

## Experimental configuration

- Model: Wan2.1-T2V-1.3B
- Implementation: Hugging Face Diffusers
- Precision: FP16 (VAE uses FP32)
- Resolution: 832 x 480
- Number of frames: 81
- Scheduler: UniPC
- Flow shift: 8.0
- Guidance scale: 6.0
- Baseline inference steps: 50
- Random seed: 42

The main experiment will vary only the number of inference steps.
All other generation parameters will remain fixed.

In [1]:
import sys
import platform
import subprocess
import torch

print("=" * 60)
print("SYSTEM")
print("=" * 60)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

print("\n" + "=" * 60)
print("AVAILABLE GPUS")
print("=" * 60)

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(f"\nGPU {i}: {props.name}")
    print("  Compute capability:", props.major, props.minor)
    print("  VRAM:", round(props.total_memory / 1024**3, 2), "GB")

    if props.major < 7 or (props.major == 7 and props.minor < 0):
        raise RuntimeError("GPU architecture is too old.")

print("\nFP16 support:", torch.cuda.is_available())
print("BF16 support:", torch.cuda.is_bf16_supported())

SYSTEM
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
PyTorch CUDA: 13.0
CUDA available: True

AVAILABLE GPUS

GPU 0: Tesla T4
  Compute capability: 7 5
  VRAM: 14.56 GB

GPU 1: Tesla T4
  Compute capability: 7 5
  VRAM: 14.56 GB

FP16 support: True
BF16 support: True


In [2]:
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

OUTPUT_DIR = WORK_DIR / "baseline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = WORK_DIR / "experiment_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORK_DIR)
print("Output directory:", OUTPUT_DIR)
print("Result directory:", RESULT_DIR)

Working directory: /kaggle/working
Output directory: /kaggle/working/baseline_outputs
Result directory: /kaggle/working/experiment_results


# Installing depencies

In [3]:
%pip install --no-cache-dir --no-deps "diffusers==0.39.0"

%pip install --no-cache-dir \
    "transformers==4.57.6" \
    "accelerate>=1.1.1" \
    "huggingface_hub>=0.34,<1.0" \
    "ftfy" \
    "imageio" \
    "imageio-ffmpeg" \
    "safetensors"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 9.4 MB/s  0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.42.0
    Uninstalling diffusers-0.42.0:
      Successfully uninstalled diffusers-0.42.0
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 723.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 248.1 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninsta

# Checking for imports

In [4]:
import torch
import numpy as np
import diffusers
import transformers
import accelerate
import huggingface_hub
import ftfy
import imageio

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("ftfy:", ftfy.__version__)
print("imageio:", imageio.__version__)

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1

print("\nAll required imports completed.")

PyTorch: 2.11.0+cu130
NumPy: 2.1.3
Diffusers: 0.39.0
Transformers: 4.57.6
Accelerate: 1.15.0
Hugging Face Hub: 0.36.2
ftfy: 6.3.1
imageio: 2.38.1

All required imports completed.


# Experimental configuration

In [5]:
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

PROMPT = (
    "A red ball rolling slowly across a wooden floor "
    "in a quiet room, realistic lighting, cinematic video."
)

SEED = 42

WIDTH = 832
HEIGHT = 480
NUM_FRAMES = 81
FPS = 16

BASELINE_STEPS = 50
GUIDANCE_SCALE = 6.0
FLOW_SHIFT = 3.0

DEVICE_ID = 0

print("Model:", MODEL_ID)
print("Prompt:", PROMPT)
print("Resolution:", WIDTH, "x", HEIGHT)
print("Frames:", NUM_FRAMES)
print("FPS:", FPS)
print("Inference steps:", BASELINE_STEPS)
print("Guidance scale:", GUIDANCE_SCALE)
print("Flow shift:", FLOW_SHIFT)
print("GPU:", torch.cuda.get_device_name(DEVICE_ID))

Model: Wan-AI/Wan2.1-T2V-1.3B-Diffusers
Prompt: A red ball rolling slowly across a wooden floor in a quiet room, realistic lighting, cinematic video.
Resolution: 832 x 480
Frames: 81
FPS: 16
Inference steps: 50
Guidance scale: 6.0
Flow shift: 3.0
GPU: Tesla T4


# Load the model

In [6]:
import gc
import torch

from diffusers import (
    AutoencoderKLWan,
    WanPipeline,
)
from diffusers.schedulers.scheduling_unipc_multistep import (
    UniPCMultistepScheduler,
)

# Load VAE in FP32 for decoding quality.
vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
)

# Load the remaining pipeline components in FP16.
pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
)

# Match the UniPC solver and flow shift used in our baseline configuration.
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=FLOW_SHIFT,
)

# Keep model components on CPU and move them to GPU when needed.
pipe.enable_model_cpu_offload(gpu_id=DEVICE_ID)

print("Model loaded successfully.")
print("Scheduler:", type(pipe.scheduler).__name__)
print("Transformer dtype:", next(pipe.transformer.parameters()).dtype)
print("VAE dtype:", next(pipe.vae.parameters()).dtype)

I0000 00:00:1795136233.751874      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/508M [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

text_encoder/model-00003-of-00005.safete(…):   0%|          | 0.00/4.97G [00:00<?, ?B/s]

... (15 outputs left out)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded successfully.
Scheduler: UniPCMultistepScheduler
Transformer dtype: torch.float32
VAE dtype: torch.float32


# Smoke test (So as to make sure everything works correctly)

In [7]:
import torch
from diffusers.utils import export_to_video

SMOKE_STEPS = 5
SMOKE_PATH = OUTPUT_DIR / "smoke_test.mp4"

generator = torch.Generator(device="cpu").manual_seed(SEED)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE_ID)

with torch.inference_mode():
    smoke_output = pipe(
        prompt=PROMPT,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=SMOKE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )

export_to_video(
    smoke_output.frames[0],
    str(SMOKE_PATH),
    fps=FPS,
)

print("Smoke test completed.")
print("Output:", SMOKE_PATH)
print("File exists:", SMOKE_PATH.exists())
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated(DEVICE_ID) / 1024**3, 2),
    "GB",
)

  0%|          | 0/5 [00:00<?, ?it/s]

Smoke test completed.
Output: /kaggle/working/baseline_outputs/smoke_test.mp4
File exists: True
Peak GPU memory: 12.68 GB


# Baseline: 50 steps

In [8]:
import time
import torch
from diffusers.utils import export_to_video

BASELINE_PATH = OUTPUT_DIR / "baseline_50_steps.mp4"

generator = torch.Generator(device="cpu").manual_seed(SEED)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE_ID)

torch.cuda.synchronize(DEVICE_ID)
start_time = time.perf_counter()

with torch.inference_mode():
    baseline_output = pipe(
        prompt=PROMPT,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=BASELINE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )

torch.cuda.synchronize(DEVICE_ID)
generation_time = time.perf_counter() - start_time

export_to_video(
    baseline_output.frames[0],
    str(BASELINE_PATH),
    fps=FPS,
)

peak_memory_allocated = (
    torch.cuda.max_memory_allocated(DEVICE_ID) / 1024**3
)

peak_memory_reserved = (
    torch.cuda.max_memory_reserved(DEVICE_ID) / 1024**3
)

print("=" * 60)
print("BASELINE RESULTS")
print("=" * 60)

print("Output:", BASELINE_PATH)
print("File exists:", BASELINE_PATH.exists())
print("Generation time:", round(generation_time, 2), "seconds")
print("Peak allocated GPU memory:", round(peak_memory_allocated, 2), "GB")
print("Peak reserved GPU memory:", round(peak_memory_reserved, 2), "GB")

  0%|          | 0/50 [00:00<?, ?it/s]

BASELINE RESULTS
Output: /kaggle/working/baseline_outputs/baseline_50_steps.mp4
File exists: True
Generation time: 2696.17 seconds
Peak allocated GPU memory: 12.71 GB
Peak reserved GPU memory: 12.83 GB


# View the resulting video

In [9]:
from IPython.display import Video, display

if not BASELINE_PATH.exists():
    raise FileNotFoundError("Baseline video was not generated.")

display(Video(str(BASELINE_PATH), embed=True))

# Save baseline metrics

In [10]:
import pandas as pd

baseline_metrics = pd.DataFrame([{
    "model": MODEL_ID,
    "prompt": PROMPT,
    "seed": SEED,
    "width": WIDTH,
    "height": HEIGHT,
    "num_frames": NUM_FRAMES,
    "fps": FPS,
    "inference_steps": BASELINE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "flow_shift": FLOW_SHIFT,
    "precision": "FP16 (VAE FP32)",
    "generation_time_seconds": generation_time,
    "peak_memory_allocated_gb": peak_memory_allocated,
    "peak_memory_reserved_gb": peak_memory_reserved,
    "output_file": str(BASELINE_PATH),
}])

baseline_metrics.to_csv(
    RESULT_DIR / "baseline_metrics.csv",
    index=False,
)

display(baseline_metrics)

,model,prompt,seed,width,height,num_frames,fps,inference_steps,guidance_scale,flow_shift,precision,generation_time_seconds,peak_memory_allocated_gb,peak_memory_reserved_gb,output_file
0,Wan-AI/Wan2.1-T2V-1.3B-Diffusers,A red ball rolling slowly across a wooden floo...,42,832,480,81,16,50,6.0,3.0,FP16 (VAE FP32),2696.170154,12.708624,12.826172,/kaggle/working/baseline_outputs/baseline_50_s...


# Run all step configurations

In [11]:
import gc
import time
import torch
import pandas as pd

from diffusers.utils import export_to_video

STEP_CONFIGS = [40, 30, 20, 10]

experiment_records = []

for steps in STEP_CONFIGS:
    print(f"\nRunning experiment with {steps} inference steps...")

    output_path = OUTPUT_DIR / f"wan_{steps}_steps.mp4"

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE_ID)

    # Reset the seed so each configuration starts from the same noise.
    generator = torch.Generator(device="cpu").manual_seed(SEED)

    torch.cuda.synchronize(DEVICE_ID)
    start_time = time.perf_counter()

    with torch.inference_mode():
        output = pipe(
            prompt=PROMPT,
            height=HEIGHT,
            width=WIDTH,
            num_frames=NUM_FRAMES,
            num_inference_steps=steps,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        )

    torch.cuda.synchronize(DEVICE_ID)
    elapsed_time = time.perf_counter() - start_time

    peak_allocated = (
        torch.cuda.max_memory_allocated(DEVICE_ID) / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(DEVICE_ID) / 1024**3
    )

    export_to_video(
        output.frames[0],
        str(output_path),
        fps=FPS,
    )

    experiment_records.append({
        "inference_steps": steps,
        "generation_time_seconds": elapsed_time,
        "peak_memory_allocated_gb": peak_allocated,
        "peak_memory_reserved_gb": peak_reserved,
        "output_file": str(output_path),
        "file_exists": output_path.exists(),
    })

    print("Time:", round(elapsed_time, 2), "seconds")
    print("Peak allocated memory:", round(peak_allocated, 2), "GB")
    print("Output:", output_path)

    del output
    gc.collect()
    torch.cuda.empty_cache()

experiment_df = pd.DataFrame(experiment_records)

experiment_df.to_csv(
    RESULT_DIR / "step_reduction_results.csv",
    index=False,
)

display(experiment_df)


Running experiment with 40 inference steps...


  0%|          | 0/40 [00:00<?, ?it/s]

Time: 2186.89 seconds
Peak allocated memory: 12.71 GB
Output: /kaggle/working/baseline_outputs/wan_40_steps.mp4

Running experiment with 30 inference steps...


  0%|          | 0/30 [00:00<?, ?it/s]

Time: 1679.8 seconds
Peak allocated memory: 12.71 GB
Output: /kaggle/working/baseline_outputs/wan_30_steps.mp4

Running experiment with 20 inference steps...


  0%|          | 0/20 [00:00<?, ?it/s]

Time: 1172.9 seconds
Peak allocated memory: 12.71 GB
Output: /kaggle/working/baseline_outputs/wan_20_steps.mp4

Running experiment with 10 inference steps...


  0%|          | 0/10 [00:00<?, ?it/s]

Time: 666.59 seconds
Peak allocated memory: 12.71 GB
Output: /kaggle/working/baseline_outputs/wan_10_steps.mp4


,inference_steps,generation_time_seconds,peak_memory_allocated_gb,peak_memory_reserved_gb,output_file,file_exists
0,40,2186.891572,12.708618,12.808594,/kaggle/working/baseline_outputs/wan_40_steps.mp4,True
1,30,1679.800930,12.708617,12.839844,/kaggle/working/baseline_outputs/wan_30_steps.mp4,True
2,20,1172.895265,12.708614,12.808594,/kaggle/working/baseline_outputs/wan_20_steps.mp4,True
3,10,666.586587,12.708622,12.824219,/kaggle/working/baseline_outputs/wan_10_steps.mp4,True


# Results

In [12]:
baseline_time = generation_time

experiment_df["speedup_vs_baseline"] = (
    baseline_time / experiment_df["generation_time_seconds"]
)

experiment_df["time_reduction_percent"] = (
    1 - experiment_df["generation_time_seconds"] / baseline_time
) * 100

display(
    experiment_df[
        [
            "inference_steps",
            "generation_time_seconds",
            "speedup_vs_baseline",
            "time_reduction_percent",
            "peak_memory_allocated_gb",
        ]
    ].sort_values("inference_steps", ascending=False)
)

,inference_steps,generation_time_seconds,speedup_vs_baseline,time_reduction_percent,peak_memory_allocated_gb
0,40,2186.891572,1.232878,18.888963,12.708618
1,30,1679.800930,1.605053,37.696776,12.708617
2,20,1172.895265,2.298731,56.497728,12.708614
3,10,666.586587,4.044741,75.276539,12.708622
